# Pràctica 4 
## Part 1: Entrenament d'Embeddings Estàtics

**Objectiu**: Entrenar models Word2Vec i fastText sobre el corpus Wikipedia en espanyol i guardar-los per ser reutilitzats a les parts 2 i 3.

## Índex
1. [Instal·lació i imports](#1-installació-i-imports)
2. [Configuració global](#2-configuració-global)
3. [Descàrrega del corpus](#3-descàrrega-del-corpus)
4. [Preprocessament del corpus](#4-preprocessament-del-corpus)
5. [Entrenament Word2Vec](#5-entrenament-word2vec)
6. [Entrenament fastText](#6-entrenament-fasttext)
7. [Càrrega del fastText oficial](#7-càrrega-del-fasttext-oficial)
8. [Resum i verificació dels models](#8-resum-i-verificació-dels-models)
9. [Funcions d'accés als embeddings (API per a Parts 2 i 3)](#9-funcions-daccés-als-embeddings-api-per-a-parts-2-i-3)

---
## 1. Instal·lació i imports

In [1]:
import os
import re
import logging
import pickle
from pathlib import Path
from typing import List, Dict, Optional, Iterator

import numpy as np
from tqdm.auto import tqdm

# Gensim: Word2Vec i fastText
from gensim.models import Word2Vec, FastText as GensimFastText
from gensim.models.callbacks import CallbackAny2Vec

# fastText oficial de Facebook (per comparació al Bloc 2)
import fasttext #  pip install fasttext-wheel

# Configura el sistema de logging para mostrar mensajes con fecha, nivel y texto.
# Crea un logger asociado al módulo actual para registrar avisos e información del proceso.
logging.basicConfig(format='%(asctime)s : %(levelname)s : %(message)s', level=logging.INFO)
logger = logging.getLogger(__name__)

c:\Natalia\Trabajo\Estudios\5_Inteligencia_Artificial\Q4\PLH\Practica-4-PLH\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---
## 2. Configuració global

Totes les rutes i hiperparàmetres estan centralitzats aquí.
**Les parts 2 i 3 importaran o faran `%run` d'aquest notebook i accediran a les constants de `CONFIG`.**

In [2]:
#  CONFIGURACIÓ GLOBAL  (llegida per les Parts 2 i 3)

CONFIG = {
    # Rutes
    "corpus_dir"        : "data/raw.es/",
    "source_encoding"  : "cp1252",
    "processed_corpus"  : "data/corpus_processed.txt",  # un token per línia, frases separades per \n
    "models_dir"        : "models/",

    # Preprocessament 
    "max_sentences"     : None,   # None = corpus complet; poseu un int per limitar (e.g. 500_000)
    "min_token_length"  : 2,

    # Hiperparàmetres Word2Vec / fastText 
    "embedding_dims"    : [25, 50, 100],   # dimensions a comparar
    "window"            : 5,
    "min_count"         : 5,
    "workers"           : 4,
    "epochs"            : 5,
    "sg"                : 1,   # 1 = Skip-gram, 0 = CBOW

    # fastText oficial
    "fasttext_official_url"  : "https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.es.300.bin.gz",
    "fasttext_official_path" : "models/cc.es.300.bin",
    "fasttext_official_dim"  : 300,

    # Seed de reproducibilitat
    "seed"              : 42,
}

# Crea directoris necessaris
for d in ["data", CONFIG["models_dir"]]:
    Path(d).mkdir(parents=True, exist_ok=True)

print("Configuració:")
for k, v in CONFIG.items():
    print(f"  {k:30s} = {v}")

Configuració:
  corpus_dir                     = data/raw.es/
  source_encoding                = cp1252
  processed_corpus               = data/corpus_processed.txt
  models_dir                     = models/
  max_sentences                  = None
  min_token_length               = 2
  embedding_dims                 = [25, 50, 100]
  window                         = 5
  min_count                      = 5
  workers                        = 4
  epochs                         = 5
  sg                             = 1
  fasttext_official_url          = https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.es.300.bin.gz
  fasttext_official_path         = models/cc.es.300.bin
  fasttext_official_dim          = 300
  seed                           = 42


In [3]:
# Inspecció: quants fitxers i quines primeres línies
corpus_files = sorted(Path(CONFIG["corpus_dir"]).rglob("*"))
corpus_files = [f for f in corpus_files if f.is_file()]
print(f"Fitxers al corpus: {len(corpus_files)}")
for f in corpus_files[:5]:
    print(f"  {f}  ({f.stat().st_size / 1e6:.1f} MB)")

# Mostra les primeres línies del primer fitxer
if corpus_files:
    with open(corpus_files[0], encoding=CONFIG["source_encoding"], errors="replace") as fh:
        for i, line in enumerate(fh):
            print(line.rstrip())
            if i >= 4:
                break

Fitxers al corpus: 57
  data\raw.es\spanishText_10000_15000  (20.3 MB)
  data\raw.es\spanishText_110000_115000  (14.6 MB)
  data\raw.es\spanishText_120000_125000  (14.5 MB)
  data\raw.es\spanishText_15000_20000  (26.5 MB)
  data\raw.es\spanishText_180000_185000  (11.5 MB)
<doc id="20540" title="658" nonfiltered="1" processed="1" dbindex="10000">

 Acontecimientos .




---
## 3. Preprocessament del corpus

Genera `corpus_processed.txt`: un fitxer amb una frase per línia i paraules en minúscules separades per espais.  
Ara també divideix les línies llargues en frases més petites abans de netejar-les, que és més útil per entrenar embeddings.
Aquest fitxer es reutilitzarà a la Part 3 per construir el vocabulari del model siamès.

In [6]:
# Regex de preprocessament
_RE_CLEAN = re.compile(r"[^a-záéíóúüñ'\-]")  # conserva lletres espanyoles
_RE_SPACES = re.compile(r"\s+")
_RE_SENT_SPLIT = re.compile(r"(?<=[.!?])\s+")


def preprocess_fragment(fragment: str) -> Optional[List[str]]:
    """
    Neteja un fragment de text i retorna la llista de tokens.
    Retorna None si el fragment és massa curt.
    """
    fragment = fragment.strip()
    if not fragment:
        return None
    fragment = fragment.lower()
    fragment = _RE_CLEAN.sub(" ", fragment)
    fragment = _RE_SPACES.sub(" ", fragment).strip()
    tokens = [t for t in fragment.split() if len(t) >= CONFIG["min_token_length"]]
    return tokens if len(tokens) >= 3 else None


def preprocess_line(line: str) -> Optional[List[List[str]]]:
    """
    Neteja una línia de text i retorna una llista de frases tokenitzades.
    Retorna None si la línia és metadada XML/wiki o no conté frases útils.
    """
    line = line.strip()
    # Descarta línies buides, XML/wiki markup i títols de seccions
    if not line or line.startswith("<") or line.startswith("="):
        return None

    sentences = []
    for fragment in _RE_SENT_SPLIT.split(line):
        tokens = preprocess_fragment(fragment)
        if tokens:
            sentences.append(tokens)

    return sentences if sentences else None


def build_processed_corpus(
    corpus_files: List[Path],
    output_path: str,
    max_sentences: Optional[int] = None,
    force_rebuild: bool = False,
) -> int:
    """
    Itera tots els fitxers del corpus, preprocessa cada línia i guarda
    el resultat com a corpus_processed.txt.

    Retorna el nombre total de frases escrites.
    """
    output_file = Path(output_path)
    if output_file.exists() and not force_rebuild:
        n = sum(1 for _ in open(output_file, encoding="utf-8"))
        print(f"El corpus processat ja existeix ({n:,} frases): {output_path}")
        return n

    if output_file.exists():
        output_file.unlink()

    total = 0
    with open(output_file, "w", encoding="utf-8") as out:
        for fpath in tqdm(corpus_files, desc="Fitxers"):
            with open(fpath, encoding=CONFIG["source_encoding"], errors="replace") as fh:
                for line in fh:
                    sentence_list = preprocess_line(line)
                    if sentence_list:
                        for tokens in sentence_list:
                            out.write(" ".join(tokens) + "\n")
                            total += 1
                            if max_sentences and total >= max_sentences:
                                print(f"  [max_sentences={max_sentences} assolit]")
                                return total
    print(f"Corpus processat: {total:,} frases -> {output_path}")
    return total


n_sentences = build_processed_corpus(
    corpus_files,
    CONFIG["processed_corpus"],
    max_sentences=CONFIG["max_sentences"],
    force_rebuild=False,  # Posem True per forçar la reconstrucció i assegurar que tenim el corpus actualitzat
)
print(f"Total de frases: {n_sentences:,}")

Fitxers: 100%|██████████| 57/57 [01:45<00:00,  1.85s/it]

Corpus processat: 5,461,482 frases -> data/corpus_processed.txt
Total de frases: 5,461,482


In [7]:
class SentenceIterator:
    """
    Iterador perezós sobre corpus_processed.txt.
    Reutilitzat per Word2Vec, fastText i la construcció del vocabulari (Part 3).

    Ús:
        sentences = SentenceIterator(CONFIG["processed_corpus"])
        for tokens in sentences:
            ...
    """
    def __init__(self, path: str):
        self.path = path

    def __iter__(self) -> Iterator[List[str]]:
        with open(self.path, encoding="utf-8") as fh:
            for line in fh:
                tokens = line.rstrip().split()
                if tokens:
                    yield tokens


# Verificació ràpida
sentences = SentenceIterator(CONFIG["processed_corpus"])
for i, s in enumerate(sentences):
    print(s)
    if i >= 3:
        break

['fulgencio', 'de', 'écija', 'santo', 'español']
['erquinoaldo', 'mayordomo', 'franco', 'de', 'palacio', 'de', 'neustria']
['egilona', 'última', 'reina', 'visigoda', 'de', 'hispania']
['fin', 'del', 'califato', 'perfecto']


---
## 5. Entrenament Word2Vec

Entrena un model per cada dimensió definida a `CONFIG["embedding_dims"]`.  
Els models es guarden com `models/w2v_dim{d}.model` i es referencien al diccionari `W2V_MODELS`.

In [8]:
class EpochLogger(CallbackAny2Vec):
    """Callback que imprimeix la pèrdua al final de cada època."""
    def __init__(self, label: str):
        self.label = label
        self.epoch = 0
        self._prev_loss = 0

    def on_epoch_end(self, model):
        loss = model.get_latest_training_loss()
        delta = loss - self._prev_loss
        self._prev_loss = loss
        self.epoch += 1
        print(f"  [{self.label}] Època {self.epoch}/{CONFIG['epochs']}  "
              f"loss acumulada={loss:.0f}  Δ={delta:.0f}")


def train_word2vec(dim: int, sentences: SentenceIterator, force_retrain: bool = False) -> Word2Vec:
    """
    Entrena o carrega un model Word2Vec de dimensió `dim`.

    Args:
        dim: dimensió dels vectors d'embedding.
        sentences: iterador sobre el corpus processat.
        force_retrain: si True, reentrena el model inclús si ja existeix. Si False (per defecte),
                       carrega el model existent si existeix.

    Returns:
        Model Word2Vec de gensim.
    """
    model_path = Path(CONFIG["models_dir"]) / f"w2v_dim{dim}.model"
    
    if model_path.exists() and not force_retrain:
        print(f"[OK] Carregant W2V dim={dim} des de {model_path}")
        return Word2Vec.load(str(model_path))
    
    if model_path.exists() and force_retrain:
        print(f"[INFO] force_retrain=True: esborrant model anterior: {model_path}")
        model_path.unlink()

    print(f"\n── Entrenant Word2Vec dim={dim} ──")
    model = Word2Vec(
        sentences=sentences,
        vector_size=dim,
        window=CONFIG["window"],
        min_count=CONFIG["min_count"],
        workers=CONFIG["workers"],
        epochs=CONFIG["epochs"],
        sg=CONFIG["sg"],
        seed=CONFIG["seed"],
        compute_loss=True,
        callbacks=[EpochLogger(f"W2V-{dim}")],
    )
    model.save(str(model_path))
    print(f"[OK] Model guardat: {model_path}")
    return model


# ── Entrenar tots els W2V ─────────────────────────────────────────────────
# Diccionari reutilitzat a Parts 2 i 3
# Clau: "w2v_25", "w2v_50", "w2v_100"
# Canvia force_retrain=False a force_retrain=True si vols reentre nar els models
W2V_MODELS: Dict[str, Word2Vec] = {}

FORCE_RETRAIN_W2V = False  # Canvia a True per reentre nar els models Word2Vec

for dim in CONFIG["embedding_dims"]:
    key = f"w2v_{dim}"
    W2V_MODELS[key] = train_word2vec(dim, SentenceIterator(CONFIG["processed_corpus"]), force_retrain=FORCE_RETRAIN_W2V)
    vocab_size = len(W2V_MODELS[key].wv)
    print(f"  Vocabulari {key}: {vocab_size:,} paraules  |  dim={dim}")

print("\nModels Word2Vec entrenats:", list(W2V_MODELS.keys()))

2026-05-20 10:39:26,272 : INFO : loading Word2Vec object from models\w2v_dim25.model


[OK] Carregant W2V dim=25 des de models\w2v_dim25.model


2026-05-20 10:39:26,497 : INFO : loading wv recursively from models\w2v_dim25.model.wv.* with mmap=None
2026-05-20 10:39:26,497 : INFO : setting ignored attribute cum_table to None
2026-05-20 10:39:28,445 : INFO : Word2Vec lifecycle event {'fname': 'models\\w2v_dim25.model', 'datetime': '2026-05-20T10:39:28.445226', 'gensim': '4.4.0', 'python': '3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]', 'platform': 'Windows-10-10.0.19045-SP0', 'event': 'loaded'}
2026-05-20 10:39:28,446 : INFO : loading Word2Vec object from models\w2v_dim50.model
2026-05-20 10:39:28,588 : INFO : loading wv recursively from models\w2v_dim50.model.wv.* with mmap=None
2026-05-20 10:39:28,589 : INFO : loading vectors from models\w2v_dim50.model.wv.vectors.npy with mmap=None


  Vocabulari w2v_25: 300,250 paraules  |  dim=25
[OK] Carregant W2V dim=50 des de models\w2v_dim50.model


2026-05-20 10:39:28,669 : INFO : loading syn1neg from models\w2v_dim50.model.syn1neg.npy with mmap=None
2026-05-20 10:39:28,749 : INFO : setting ignored attribute cum_table to None
2026-05-20 10:39:30,690 : INFO : Word2Vec lifecycle event {'fname': 'models\\w2v_dim50.model', 'datetime': '2026-05-20T10:39:30.690764', 'gensim': '4.4.0', 'python': '3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]', 'platform': 'Windows-10-10.0.19045-SP0', 'event': 'loaded'}
2026-05-20 10:39:30,692 : INFO : loading Word2Vec object from models\w2v_dim100.model
2026-05-20 10:39:30,835 : INFO : loading wv recursively from models\w2v_dim100.model.wv.* with mmap=None
2026-05-20 10:39:30,837 : INFO : loading vectors from models\w2v_dim100.model.wv.vectors.npy with mmap=None


  Vocabulari w2v_50: 300,250 paraules  |  dim=50
[OK] Carregant W2V dim=100 des de models\w2v_dim100.model


2026-05-20 10:39:31,038 : INFO : loading syn1neg from models\w2v_dim100.model.syn1neg.npy with mmap=None
2026-05-20 10:39:31,236 : INFO : setting ignored attribute cum_table to None
2026-05-20 10:39:33,186 : INFO : Word2Vec lifecycle event {'fname': 'models\\w2v_dim100.model', 'datetime': '2026-05-20T10:39:33.186601', 'gensim': '4.4.0', 'python': '3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]', 'platform': 'Windows-10-10.0.19045-SP0', 'event': 'loaded'}


  Vocabulari w2v_100: 300,250 paraules  |  dim=100

Models Word2Vec entrenats: ['w2v_25', 'w2v_50', 'w2v_100']


---
## 6. Entrenament fastText

Igual que Word2Vec però amb l'arquitectura fastText de Gensim (subword embeddings).  
Els models es guarden com `models/ft_dim{d}.model` i es referencien al diccionari `FT_MODELS`.

In [9]:
def train_fasttext(dim: int, sentences: SentenceIterator, force_retrain: bool = False) -> GensimFastText:
    """
    Entrena o carrega un model fastText (gensim) de dimensió `dim`.

    Args:
        dim: dimensió dels vectors d'embedding.
        sentences: iterador sobre el corpus processat.
        force_retrain: si True, reentrena el model inclús si ja existeix. Si False (per defecte),
                       carrega el model existent si existeix.

    Returns:
        Model FastText de gensim.
    """
    model_path = Path(CONFIG["models_dir"]) / f"ft_dim{dim}.model"
    
    if model_path.exists() and not force_retrain:
        print(f"[OK] Carregant FT dim={dim} des de {model_path}")
        return GensimFastText.load(str(model_path))
    
    if model_path.exists() and force_retrain:
        print(f"[INFO] force_retrain=True: esborrant model anterior: {model_path}")
        model_path.unlink()

    print(f"\n── Entrenant fastText (gensim) dim={dim} ──")
    model = GensimFastText(
        sentences=sentences,
        vector_size=dim,
        window=CONFIG["window"],
        min_count=CONFIG["min_count"],
        workers=CONFIG["workers"],
        epochs=CONFIG["epochs"],
        sg=CONFIG["sg"],
        seed=CONFIG["seed"],
    )
    model.save(str(model_path))
    print(f"[OK] Model guardat: {model_path}")
    return model


# ── Entrenar tots els FT ──────────────────────────────────────────────────
# Diccionari reutilitzat a Parts 2 i 3
# Clau: "ft_25", "ft_50", "ft_100"
# Canvia force_retrain=False a force_retrain=True si vols reentre nar els models
FT_MODELS: Dict[str, GensimFastText] = {}

FORCE_RETRAIN_FT = False  # Canvia a True per reentre nar els models fastText

for dim in CONFIG["embedding_dims"]:
    key = f"ft_{dim}"
    FT_MODELS[key] = train_fasttext(dim, SentenceIterator(CONFIG["processed_corpus"]), force_retrain=FORCE_RETRAIN_FT)
    vocab_size = len(FT_MODELS[key].wv)
    print(f"  Vocabulari {key}: {vocab_size:,} paraules  |  dim={dim}")

print("\nModels fastText (gensim) entrenats:", list(FT_MODELS.keys()))

2026-05-20 10:39:41,085 : INFO : loading FastText object from models\ft_dim25.model


[OK] Carregant FT dim=25 des de models\ft_dim25.model


2026-05-20 10:39:41,266 : INFO : loading wv recursively from models\ft_dim25.model.wv.* with mmap=None
2026-05-20 10:39:41,266 : INFO : loading vectors_ngrams from models\ft_dim25.model.wv.vectors_ngrams.npy with mmap=None
2026-05-20 10:39:41,508 : INFO : setting ignored attribute vectors to None
2026-05-20 10:39:41,508 : INFO : setting ignored attribute buckets_word to None
2026-05-20 10:39:54,428 : INFO : setting ignored attribute cum_table to None
2026-05-20 10:39:56,367 : INFO : FastText lifecycle event {'fname': 'models\\ft_dim25.model', 'datetime': '2026-05-20T10:39:56.367309', 'gensim': '4.4.0', 'python': '3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]', 'platform': 'Windows-10-10.0.19045-SP0', 'event': 'loaded'}
2026-05-20 10:39:56,368 : INFO : loading FastText object from models\ft_dim50.model
2026-05-20 10:39:56,491 : INFO : loading wv recursively from models\ft_dim50.model.wv.* with mmap=None
2026-05-20 10:39:56,491 : INFO : loading vectors_

  Vocabulari ft_25: 300,250 paraules  |  dim=25
[OK] Carregant FT dim=50 des de models\ft_dim50.model


2026-05-20 10:39:56,583 : INFO : loading vectors_ngrams from models\ft_dim50.model.wv.vectors_ngrams.npy with mmap=None
2026-05-20 10:39:57,194 : INFO : setting ignored attribute vectors to None
2026-05-20 10:39:57,194 : INFO : setting ignored attribute buckets_word to None
2026-05-20 10:40:10,148 : INFO : loading syn1neg from models\ft_dim50.model.syn1neg.npy with mmap=None
2026-05-20 10:40:10,251 : INFO : setting ignored attribute cum_table to None
2026-05-20 10:40:12,199 : INFO : FastText lifecycle event {'fname': 'models\\ft_dim50.model', 'datetime': '2026-05-20T10:40:12.199075', 'gensim': '4.4.0', 'python': '3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]', 'platform': 'Windows-10-10.0.19045-SP0', 'event': 'loaded'}
2026-05-20 10:40:12,199 : INFO : loading FastText object from models\ft_dim100.model
2026-05-20 10:40:12,305 : INFO : loading wv recursively from models\ft_dim100.model.wv.* with mmap=None
2026-05-20 10:40:12,305 : INFO : loading vector

  Vocabulari ft_50: 300,250 paraules  |  dim=50
[OK] Carregant FT dim=100 des de models\ft_dim100.model


2026-05-20 10:40:12,453 : INFO : loading vectors_ngrams from models\ft_dim100.model.wv.vectors_ngrams.npy with mmap=None
2026-05-20 10:40:13,403 : INFO : setting ignored attribute vectors to None
2026-05-20 10:40:13,403 : INFO : setting ignored attribute buckets_word to None
2026-05-20 10:40:26,413 : INFO : loading syn1neg from models\ft_dim100.model.syn1neg.npy with mmap=None
2026-05-20 10:40:26,552 : INFO : setting ignored attribute cum_table to None
2026-05-20 10:40:28,469 : INFO : FastText lifecycle event {'fname': 'models\\ft_dim100.model', 'datetime': '2026-05-20T10:40:28.469196', 'gensim': '4.4.0', 'python': '3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]', 'platform': 'Windows-10-10.0.19045-SP0', 'event': 'loaded'}


  Vocabulari ft_100: 300,250 paraules  |  dim=100

Models fastText (gensim) entrenats: ['ft_25', 'ft_50', 'ft_100']


---
## 7. Càrrega del fastText oficial (Facebook)

El model oficial `cc.es.300.bin` s'usa a la Part 2 per comparar amb els models propis.  
Per la seva mida (~7 GB), es descarrega per separat. Si no el tens, la variable `FT_OFFICIAL` serà `None` i les parts posteriors ho gestionen.

In [15]:
def download_fasttext_official(
    url: str, dest: str, force: bool = False
) -> None:
    """
    Descarrega el model fastText oficial (cc.es.300.bin.gz) i el descomprimeix.
    Si dest ja existeix, no fa res (llevat que force=True).
    """
    dest_path = Path(dest)
    gz_path = dest_path.with_suffix(".bin.gz")
    if dest_path.exists() and not force:
        print(f"[OK] fastText oficial ja descarregat: {dest}")
        return
    print(f"Descarregant fastText oficial (~7 GB) ...")
    os.system(f"wget -q --show-progress -O '{gz_path}' '{url}'")
    os.system(f"gunzip '{gz_path}'")
    print(f"[OK] Guardat a {dest}")


def load_fasttext_official(path: str):
    """
    Carrega el model fastText oficial amb la llibreria `fasttext`.
    Retorna el model o None si el fitxer no existeix o fasttext no és disponible.
    """
    if not Path(path).exists():
        print(f"[AVÍS] Model oficial no trobat: {path}")
        print("       Descomenta la línia de descàrrega o descarrega'l manualment.")
        return None
    print(f"Carregant fastText oficial des de {path} ...")
    model = fasttext.load_model(path)
    print(f"[OK] Model carregat. Dim={model.get_dimension()}")
    return model


# Descomenta per descarregar (~7 GB):
# download_fasttext_official(
#     CONFIG["fasttext_official_url"],
#     CONFIG["fasttext_official_path"]
# )

# Variable reutilitzada a la Part 2
FT_OFFICIAL = load_fasttext_official(CONFIG["fasttext_official_path"])

[AVÍS] Model oficial no trobat: models/cc.es.300.bin
       Descomenta la línia de descàrrega o descarrega'l manualment.


---
## 8. Resum i verificació dels models

In [11]:
# ── Resum de tots els models disponibles ─────────────────────────────────
print("=" * 55)
print("MODELS ENTRENATS")
print("=" * 55)

all_models_info = []

for key, model in W2V_MODELS.items():
    dim = model.vector_size
    vocab = len(model.wv)
    path = Path(CONFIG["models_dir"]) / f"w2v_dim{dim}.model"
    size_mb = path.stat().st_size / 1e6 if path.exists() else 0
    all_models_info.append((key, "Word2Vec", dim, vocab, size_mb))

for key, model in FT_MODELS.items():
    dim = model.vector_size
    vocab = len(model.wv)
    path = Path(CONFIG["models_dir"]) / f"ft_dim{dim}.model"
    size_mb = path.stat().st_size / 1e6 if path.exists() else 0
    all_models_info.append((key, "fastText", dim, vocab, size_mb))

if FT_OFFICIAL:
    dim = FT_OFFICIAL.get_dimension()
    path = Path(CONFIG["fasttext_official_path"])
    size_mb = path.stat().st_size / 1e6 if path.exists() else 0
    all_models_info.append(("ft_official", "fastText oficial", dim, "—", size_mb))

print(f"{'Clau':<15} {'Tipus':<18} {'Dim':>5} {'Vocab':>10} {'Mida (MB)':>10}")
print("-" * 55)
for clau, tipus, dim, vocab, mida in all_models_info:
    vocab_str = f"{vocab:,}" if isinstance(vocab, int) else vocab
    print(f"{clau:<15} {tipus:<18} {dim:>5} {vocab_str:>10} {mida:>9.1f}")

MODELS ENTRENATS
Clau            Tipus                Dim      Vocab  Mida (MB)
-------------------------------------------------------
w2v_25          Word2Vec              25    300,250      69.9
w2v_50          Word2Vec              50    300,250       9.9
w2v_100         Word2Vec             100    300,250       9.9
ft_25           fastText              25    300,250      69.9
ft_50           fastText              50    300,250       9.9
ft_100          fastText             100    300,250       9.9


In [12]:
# ── Test de similituds (verificació bàsica) ───────────────────────────────
TEST_WORDS = [("rey", "reina"), ("perro", "gato"), ("madrid", "barcelona")]

print("Test de similitud cosinus (verificació):")
print(f"  {'Model':<12} {'Par':<22} {'Sim':>6}")
print("  " + "-" * 42)

for key, model in {**W2V_MODELS, **FT_MODELS}.items():
    for w1, w2 in TEST_WORDS:
        try:
            sim = model.wv.similarity(w1, w2)
            print(f"  {key:<12} {w1:>9} — {w2:<9}  {sim:>6.3f}")
        except KeyError:
            print(f"  {key:<12} {w1:>9} — {w2:<9}  [OOV]")

Test de similitud cosinus (verificació):
  Model        Par                       Sim
  ------------------------------------------
  w2v_25             rey — reina       0.822
  w2v_25           perro — gato        0.933
  w2v_25          madrid — barcelona   0.927
  w2v_50             rey — reina       0.781
  w2v_50           perro — gato        0.903
  w2v_50          madrid — barcelona   0.901
  w2v_100            rey — reina       0.697
  w2v_100          perro — gato        0.831
  w2v_100         madrid — barcelona   0.816
  ft_25              rey — reina       0.828
  ft_25            perro — gato        0.943
  ft_25           madrid — barcelona   0.927
  ft_50              rey — reina       0.778
  ft_50            perro — gato        0.914
  ft_50           madrid — barcelona   0.901
  ft_100             rey — reina       0.692
  ft_100           perro — gato        0.836
  ft_100          madrid — barcelona   0.802


---
## 9. Funcions d'accés als embeddings (API per a Parts 2 i 3)

Definim aquí funcions estables que **les parts 2 i 3 importaran directament**.
No cal saber quina és la implementació interna de cada model.

In [13]:
def get_word_vector(
    word: str,
    model_key: str,
) -> Optional[np.ndarray]:
    """
    Retorna el vector d'una paraula per a un model determinat.

    Args:
        word:      paraula a consultar.
        model_key: identificador del model: "w2v_25", "w2v_50", "w2v_100",
                   "ft_25", "ft_50", "ft_100", o "ft_official".

    Returns:
        np.ndarray de forma (dim,) o None si la paraula no es troba.
    """
    if model_key == "ft_official":
        if FT_OFFICIAL is None:
            return None
        return FT_OFFICIAL.get_word_vector(word).astype(np.float32)

    if model_key in W2V_MODELS:
        model = W2V_MODELS[model_key]
        if word in model.wv:
            return model.wv[word]
        return None  # W2V no genera vectors per OOV

    if model_key in FT_MODELS:
        model = FT_MODELS[model_key]
        # fastText genera vectors per qualsevol paraula (subwords)
        return model.wv.get_vector(word, norm=False)

    raise ValueError(f"Model desconegut: '{model_key}'. "
                     f"Opcions: {list(W2V_MODELS) + list(FT_MODELS) + ['ft_official']}")


def get_sentence_vector(
    tokens: List[str],
    model_key: str,
    weights: Optional[np.ndarray] = None,
) -> Optional[np.ndarray]:
    """
    Representa una frase com la mitjana (simple o ponderada) dels seus vectors.
    Usada al baseline cosinus de la Part 3.

    Args:
        tokens:    llista de paraules de la frase (ja tokenitzades).
        model_key: identificador del model (vegeu get_word_vector).
        weights:   pesos TF-IDF per a cada token (mateixa longitud que tokens).
                   Si None, es fa la mitjana simple.

    Returns:
        Vector np.ndarray de forma (dim,), o None si cap token té vector.
    """
    vecs, ws = [], []
    for i, token in enumerate(tokens):
        v = get_word_vector(token, model_key)
        if v is not None:
            vecs.append(v)
            ws.append(weights[i] if weights is not None else 1.0)

    if not vecs:
        return None

    vecs_arr = np.array(vecs)   # (n_tokens, dim)
    ws_arr = np.array(ws, dtype=np.float32).reshape(-1, 1)  # (n_tokens, 1)
    ws_arr = ws_arr / (ws_arr.sum() + 1e-9)                  # normalitza
    return (vecs_arr * ws_arr).sum(axis=0)


def build_embedding_matrix(
    vocab: Dict[str, int],
    model_key: str,
    dim: int,
) -> np.ndarray:
    """
    Construeix la matriu d'embeddings per al model siamès BiLSTM (Part 3).

    Índex 0 → <PAD>  (vector de zeros)
    Índex 1 → <UNK>  (vector aleatori o de zeros)
    Índex i → vector de la i-èssima paraula del vocabulari

    Args:
        vocab:     diccionari {paraula: índex} (índex 0 = PAD, 1 = UNK).
        model_key: model d'embeddings a usar.
        dim:       dimensió dels embeddings.

    Returns:
        np.ndarray de forma (|vocab|, dim).
    """
    rng = np.random.default_rng(CONFIG["seed"])
    matrix = np.zeros((len(vocab), dim), dtype=np.float32)

    # Índex 1 (<UNK>) = vector aleatori petit
    matrix[1] = rng.normal(scale=0.01, size=dim)

    n_found, n_oov = 0, 0
    for word, idx in vocab.items():
        if idx <= 1:  # PAD i UNK ja inicialitzats
            continue
        v = get_word_vector(word, model_key)
        if v is not None:
            matrix[idx] = v
            n_found += 1
        else:
            matrix[idx] = rng.normal(scale=0.01, size=dim)
            n_oov += 1

    print(f"[{model_key}] Matriu d'embeddings: {matrix.shape}  "
          f"cobertura={n_found/(n_found+n_oov+1e-9):.1%}  OOV={n_oov:,}")
    return matrix


def is_oov(word: str, model_key: str) -> bool:
    """
    Comprova si una paraula és OOV per a un model Word2Vec (FT sempre retorna vec).
    Útil per a l'anàlisi OOV de la Part 2.
    """
    if model_key in FT_MODELS or model_key == "ft_official":
        return False  # fastText genera vector per a qualsevol string
    if model_key in W2V_MODELS:
        return word not in W2V_MODELS[model_key].wv
    raise ValueError(f"Model desconegut: {model_key}")


# ── Prova de les funcions ─────────────────────────────────────────────────
print("Prova get_word_vector:")
v = get_word_vector("gato", "w2v_100")
print(f"  'gato' w2v_100 → {v[:5] if v is not None else 'OOV'}...")

print("\nProva get_sentence_vector:")
sv = get_sentence_vector(["el", "gato", "come"], "ft_100")
print(f"  'el gato come' ft_100 → {sv[:5] if sv is not None else 'Error'}...")

print("\nFuncions d'accés als embeddings llestes.")

Prova get_word_vector:
  'gato' w2v_100 → [0.15949443 0.07745793 0.14787205 0.39554453 0.03580431]...

Prova get_sentence_vector:
  'el gato come' ft_100 → [-0.0375998   0.13153526 -0.21606882  0.11614558 -0.07959796]...

Funcions d'accés als embeddings llestes.


---
## Resum de variables exportades per a les Parts 2 i 3

Les parts posteriors han d'executar (o fer `%run`) aquest notebook i podran accedir a:

| Variable / Funció | Tipus | Descripció |
|---|---|---|
| `CONFIG` | `dict` | Tota la configuració global |
| `W2V_MODELS` | `dict[str, Word2Vec]` | Models Word2Vec: `w2v_25`, `w2v_50`, `w2v_100` |
| `FT_MODELS` | `dict[str, FastText]` | Models fastText (gensim): `ft_25`, `ft_50`, `ft_100` |
| `FT_OFFICIAL` | `fasttext.FastText` o `None` | Model oficial Facebook (300d) |
| `SentenceIterator` | classe | Iterador perezós del corpus processat |
| `get_word_vector(word, model_key)` | funció | Vector d'una paraula |
| `get_sentence_vector(tokens, model_key, weights?)` | funció | Vector agregat d'una frase |
| `build_embedding_matrix(vocab, model_key, dim)` | funció | Matriu per al BiLSTM |
| `is_oov(word, model_key)` | funció | Comprova si una paraula és OOV |

**Com usar-lo des d'un altre notebook:**
```python
%run practica4_part1_embeddings.ipynb
# Ara CONFIG, W2V_MODELS, FT_MODELS, etc. estan disponibles
```